In [3]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00a 0:00:01


In [4]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [7]:
import pandas as pd

train_df = pd.read_csv(
    "/kaggle/input/datasets/debarshichanda/goemotions/data/train.tsv",
    sep="\t",
    header=None,
    names=["text", "label", "id"]
)

# Keep only needed columns
train_df = train_df[["text", "label"]]

# Handle multi-label issue (take first label only)
def fix_label(x):
    try:
        return int(str(x).split(",")[0])
    except:
        return -1

train_df["label"] = train_df["label"].apply(fix_label)

# Remove bad rows
train_df = train_df[train_df["label"] != -1]

train_df["label"] = train_df["label"].astype(int)

print("Shape:", train_df.shape)
print("Classes:", train_df["label"].nunique())
print(train_df.head())

Shape: (43410, 2)
Classes: 28
                                                text  label
0  My favourite food is anything I didn't have to...     27
1  Now if he does off himself, everyone will thin...     27
2                     WHY THE FUCK IS BAYLESS ISOING      2
3                        To make her feel threatened     14
4                             Dirty Southern Wankers      3


In [8]:
# Split dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].values,
    train_df["label"].values,
    test_size=0.1,
    random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 39069
Validation size: 4341


In [9]:
train_data = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_data = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

print(train_data)
print(val_data)

Dataset({
    features: ['text', 'label'],
    num_rows: 39069
})
Dataset({
    features: ['text', 'label'],
    num_rows: 4341
})


In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [14]:
train_data = train_data.map(tokenize_function, batched=True)
val_data = val_data.map(tokenize_function, batched=True)

print(train_data[0])

Map:   0%|          | 0/39069 [00:00<?, ? examples/s]

Map:   0%|          | 0/4341 [00:00<?, ? examples/s]

{'text': 'They look the same way fighting as I do dancing', 'label': 27, 'input_ids': [101, 2027, 2298, 1996, 2168, 2126, 3554, 2004, 1045, 2079, 5613, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
# Remove original text column (Trainer doesn't need it anymore)
train_data = train_data.remove_columns(["text"])
val_data = val_data.remove_columns(["text"])

# Set format to PyTorch tensors
train_data.set_format("torch")
val_data.set_format("torch")

print(train_data)
print(val_data)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 39069
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4341
})


In [16]:
from transformers import BertForSequenceClassification

num_labels = 28  # GoEmotions cleaned classes

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

print("Model loaded with labels:", num_labels)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded with labels: 28


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_emotion_output",

    do_eval=True,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=2,
    weight_decay=0.01,

    logging_steps=50,

    save_strategy="epoch",
    save_total_limit=2,

    fp16=True,

    report_to="none"
)

In [24]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

In [25]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics
)

In [ ]:
# Start training
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,2.772038
100,2.742245
150,2.698266
200,2.630069


In [ ]:
# Save trained model
trainer.save_model("./bert_emotion_model_final")

# Save tokenizer
tokenizer.save_pretrained("./bert_emotion_model_final")

print("Model saved successfully!")